# Airline Passenger Forecasting with Google TimesFM

This notebook implements a production-style **route-level passenger demand forecasting** workflow for airlines using **Google TimesFM 2.5**.

## Industry Problem

Airlines must estimate upcoming passenger demand per route to support:

- Dynamic ticket pricing
- Aircraft allocation and fleet balancing
- Crew scheduling and staffing

Under-forecasting creates stockout-like behavior (missed revenue, sold-out routes), while over-forecasting causes poor utilization and higher operating costs.

## Forecasting Objective

From historical booking events, predict route passenger demand for the coming **7/14 days** (and produce a 14-day operational plan).


## What You Will Learn

1. How to download and validate a relational airline bookings dataset from the web.
2. How to transform booking-level records into a daily `Date x Route x Passengers` panel.
3. How to configure and run TimesFM with calendar covariates.
4. How to evaluate against baselines (`naive_last`, `seasonal7`) using rolling backtests.
5. How to convert forecasts into operations-facing outputs:
   - pricing signals
   - fleet allocation plan
   - crew schedule plan

The notebook is fully executable and writes artifacts for downstream workflows.


In [ ]:
from __future__ import annotations

import math
import os
import sqlite3
import subprocess
import zipfile
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Reproducibility
SEED = 42
np.random.seed(SEED)

# Keep runtime deterministic and quiet on machines without CUDA.
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '')
os.environ.setdefault('JAX_PLATFORMS', 'cpu')

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / 'data' / 'airline_passengers'
RAW_DIR = DATA_DIR / 'raw'
ART_DIR = PROJECT_ROOT / 'artifacts' / 'airline_passenger_timesfm'

RAW_DIR.mkdir(parents=True, exist_ok=True)
ART_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = RAW_DIR / 'travel.sqlite'

print('Project root:', PROJECT_ROOT)
print('DB path:', DB_PATH)
print('Artifacts dir:', ART_DIR)

## 1) Web Data Download

Dataset: **Kaggle - `saadharoon27/airlines-dataset`**

Why this dataset:

- Production-like relational schema (`bookings`, `tickets`, `ticket_flights`, `flights`)
- Transaction-level booking history
- Route-level demand can be built from real joins instead of synthetic aggregates

If `travel.sqlite` is already present, the download step is skipped.


In [ ]:
def ensure_dataset(db_path: Path) -> Path:
    """Ensure travel.sqlite exists locally, downloading from Kaggle if needed."""
    if db_path.exists():
        print(f'Found existing file: {db_path}')
        return db_path

    zip_path = RAW_DIR / 'airlines-dataset.zip'
    cmd = [
        'kaggle',
        'datasets',
        'download',
        '-d',
        'saadharoon27/airlines-dataset',
        '-p',
        str(RAW_DIR),
        '--force',
    ]

    try:
        print('Downloading dataset from Kaggle...')
        subprocess.run(cmd, check=True, capture_output=True, text=True)
    except Exception as exc:
        raise RuntimeError(
            'Failed to download dataset from Kaggle. Configure Kaggle API credentials, '
            f'or place travel.sqlite at {db_path}.'
        ) from exc

    if zip_path.exists():
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(RAW_DIR)

    if not db_path.exists():
        raise FileNotFoundError(f'Expected file not found after download: {db_path}')

    return db_path


db_path = ensure_dataset(DB_PATH)
db_path

## 2) Build `Date x Route x Passengers` from Relational Tables

Demand is built by joining:

- `bookings` -> booking timestamp
- `tickets` -> booking reference
- `ticket_flights` -> ticket to flight linkage
- `flights` -> departure/arrival airports

Then aggregate to daily route passengers:

- `date`: booking date
- `route`: `departure_airport-arrival_airport`
- `passengers`: count of booked passenger-flight records


In [ ]:
sql_route_daily = """
SELECT
  substr(b.book_date, 1, 10) AS booking_date,
  f.departure_airport || '-' || f.arrival_airport AS route,
  COUNT(*) AS passengers
FROM bookings b
JOIN tickets t ON t.book_ref = b.book_ref
JOIN ticket_flights tf ON tf.ticket_no = t.ticket_no
JOIN flights f ON f.flight_id = tf.flight_id
GROUP BY 1, 2
ORDER BY 1, 2
"""

with sqlite3.connect(db_path) as conn:
    tables = pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;",
        conn,
    )
    route_daily = pd.read_sql_query(sql_route_daily, conn)

print('Tables in SQLite:', tables['name'].tolist())

route_daily['booking_date'] = pd.to_datetime(route_daily['booking_date'])
route_daily['passengers'] = route_daily['passengers'].astype(np.float32)

print('Aggregated rows:', len(route_daily))
print('Date range:', route_daily['booking_date'].min(), '->', route_daily['booking_date'].max())
print('Unique routes:', route_daily['route'].nunique())
print('Null counts:')
print(route_daily.isna().sum())

route_daily.head()

In [ ]:
# Data quality checks

dups = route_daily.duplicated(subset=['booking_date', 'route']).sum()
negative_counts = int((route_daily['passengers'] < 0).sum())

print('Duplicate (date, route) rows:', dups)
print('Negative passenger rows:', negative_counts)

if dups > 0:
    raise ValueError('Duplicate (booking_date, route) rows found after aggregation.')
if negative_counts > 0:
    raise ValueError('Negative passenger counts found.')

# Keep top-N routes by total volume for stable and fast tutorial execution.
route_volume = (
    route_daily.groupby('route', as_index=False)['passengers']
    .sum()
    .sort_values('passengers', ascending=False)
)
TOP_ROUTES = 30
selected_routes = route_volume.head(TOP_ROUTES)['route'].tolist()

print(f'Selected top routes: {len(selected_routes)}')
route_volume.head(10)

In [ ]:
# Build regular daily panel for selected routes.
route_top = route_daily[route_daily['route'].isin(selected_routes)].copy()
all_days = pd.date_range(route_top['booking_date'].min(), route_top['booking_date'].max(), freq='D')
full_idx = pd.MultiIndex.from_product([selected_routes, all_days], names=['route', 'booking_date'])

panel = (
    route_top.set_index(['route', 'booking_date'])['passengers']
    .reindex(full_idx)
    .fillna(0.0)
    .rename('passengers')
    .reset_index()
)

pivot = panel.pivot(index='booking_date', columns='route', values='passengers').sort_index()

print('Panel shape:', panel.shape)
print('Pivot shape (days x routes):', pivot.shape)
print('Date coverage:', pivot.index.min(), '->', pivot.index.max())

panel.head()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
for route in pivot.columns[:8]:
    ax.plot(pivot.index, pivot[route], lw=0.9, alpha=0.8, label=route)
ax.set_title('Sample Route-Level Daily Passenger Demand')
ax.set_ylabel('Passengers')
ax.set_xlabel('Date')
ax.grid(alpha=0.2)
ax.legend(ncol=4, fontsize=8)
plt.tight_layout()
plt.show()

## 3) Forecast Configuration

We use rolling backtests to avoid leakage:

- `context_len=21` days history used for each prediction
- `max_horizon=14` days forecast window
- evaluation horizons: `1, 7, 14` days
- anchor offsets are evaluated only if they are valid for all slicing constraints

Baselines:

- `naive_last`: repeat last observed value
- `seasonal7`: repeat last 7-day seasonal pattern


In [ ]:
@dataclass
class Config:
    context_len: int = 21
    max_horizon: int = 14
    eval_horizons: tuple[int, ...] = (1, 7, 14)
    anchor_offsets_days: tuple[int, ...] = (21, 14, 7)
    per_core_batch_size: int = 8
    xreg_mode: str = 'xreg + timesfm'
    xreg_ridge: float = 1e-3


cfg = Config()

anchor_positions: list[int] = []
for offset in cfg.anchor_offsets_days:
    end_pos = len(pivot) - 1 - offset
    if end_pos - cfg.context_len + 1 < 0:
        continue
    if end_pos + cfg.max_horizon >= len(pivot):
        continue

    valid_series = 0
    for route in pivot.columns:
        context = pivot[route].iloc[end_pos - cfg.context_len + 1 : end_pos + 1].to_numpy(np.float32)
        future = pivot[route].iloc[end_pos + 1 : end_pos + cfg.max_horizon + 1].to_numpy(np.float32)
        if len(context) == cfg.context_len and len(future) == cfg.max_horizon:
            valid_series += 1

    if valid_series > 0:
        anchor_positions.append(end_pos)

if not anchor_positions:
    raise RuntimeError('No valid backtest anchors found. Increase data window or reduce context/horizon.')

print('Valid anchors:', len(anchor_positions))
for pos in anchor_positions:
    print(' -', pivot.index[pos].date())

## 4) Load TimesFM + Forecast Helpers

We use `TimesFM 2.5 200M (PyTorch)` and compile with quantile output enabled.

We pass daily calendar covariates:

- day-of-week cyclic terms (`sin/cos`)
- weekend flag
- day-of-month scaled


In [ ]:
import timesfm

model = timesfm.TimesFM_2p5_200M_torch.from_pretrained('google/timesfm-2.5-200m-pytorch')
forecast_config = timesfm.ForecastConfig(
    max_context=cfg.context_len,
    max_horizon=cfg.max_horizon,
    normalize_inputs=True,
    per_core_batch_size=cfg.per_core_batch_size,
    use_continuous_quantile_head=True,
    force_flip_invariance=True,
    infer_is_positive=True,
    fix_quantile_crossing=True,
    return_backcast=True,
)
model.compile(forecast_config)
print('TimesFM compiled.')

In [ ]:
def wmape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.abs(y_true - y_pred).sum() / (np.abs(y_true).sum() + 1e-8))


def build_daily_covariates(start_date: pd.Timestamp, context_len: int, horizon: int) -> dict[str, list[list[float]]]:
    full_dates = pd.date_range(start_date, periods=context_len + horizon, freq='D')
    dow = full_dates.dayofweek.values.astype(np.float32)
    dom = full_dates.day.values.astype(np.float32)
    is_weekend = (dow >= 5).astype(np.float32)

    return {
        'dow_sin': [np.sin(2 * np.pi * dow / 7.0).astype(np.float32).tolist()],
        'dow_cos': [np.cos(2 * np.pi * dow / 7.0).astype(np.float32).tolist()],
        'is_weekend': [is_weekend.tolist()],
        'dom_scaled': [(dom / 31.0).astype(np.float32).tolist()],
    }


def run_timesfm_forecast(context: np.ndarray, start_date: pd.Timestamp, horizon: int) -> tuple[np.ndarray, np.ndarray]:
    cov = build_daily_covariates(start_date=start_date, context_len=len(context), horizon=horizon)

    try:
        point, quant = model.forecast_with_covariates(
            inputs=[context.astype(np.float32)],
            dynamic_numerical_covariates=cov,
            xreg_mode=cfg.xreg_mode,
            ridge=cfg.xreg_ridge,
        )
    except Exception:
        # Fallback path keeps notebook robust if xreg call fails on some environments.
        point, quant = model.forecast(horizon=horizon, inputs=[context.astype(np.float32)])

    p = np.asarray(point, dtype=np.float32)[0, :horizon]
    q = np.asarray(quant, dtype=np.float32)[0, :horizon, :]

    return np.clip(p, 0.0, None), np.clip(q, 0.0, None)

## 5) Rolling Backtest: TimesFM vs Baselines

Evaluation is computed across all selected routes and valid anchors.

Metrics:

- MAE
- RMSE
- WMAPE


In [ ]:
metric_rows: list[dict] = []

for anchor_pos in anchor_positions:
    anchor_date = pivot.index[anchor_pos]

    for route in pivot.columns:
        context = pivot[route].iloc[anchor_pos - cfg.context_len + 1 : anchor_pos + 1].to_numpy(np.float32)
        future = pivot[route].iloc[anchor_pos + 1 : anchor_pos + cfg.max_horizon + 1].to_numpy(np.float32)

        if len(context) != cfg.context_len or len(future) != cfg.max_horizon:
            continue

        start_date = pivot.index[anchor_pos - cfg.context_len + 1]
        tfm_pred, _ = run_timesfm_forecast(context=context, start_date=start_date, horizon=cfg.max_horizon)

        naive_last = np.repeat(context[-1], cfg.max_horizon)
        seasonal7 = np.tile(context[-7:], math.ceil(cfg.max_horizon / 7))[: cfg.max_horizon]

        model_map = {
            'timesfm': tfm_pred,
            'naive_last': naive_last,
            'seasonal7': seasonal7,
        }

        for hz in cfg.eval_horizons:
            y = future[:hz]
            for model_name, pred in model_map.items():
                p = pred[:hz]
                metric_rows.append(
                    {
                        'anchor_date': anchor_date,
                        'route': route,
                        'horizon_days': hz,
                        'model': model_name,
                        'mae': mean_absolute_error(y, p),
                        'rmse': mean_squared_error(y, p) ** 0.5,
                        'wmape': wmape(y, p),
                    }
                )

backtest_detail = pd.DataFrame(metric_rows)
backtest_metrics = (
    backtest_detail
    .groupby(['horizon_days', 'model'], as_index=False)[['mae', 'rmse', 'wmape']]
    .mean()
    .sort_values(['horizon_days', 'rmse'])
)

backtest_metrics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric in zip(axes, ['mae', 'rmse', 'wmape']):
    pivot_metric = backtest_metrics.pivot(index='horizon_days', columns='model', values=metric)
    pivot_metric.plot(kind='bar', ax=ax)
    ax.set_title(metric.upper())
    ax.set_xlabel('Horizon (days)')
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

## 6) Forecast Next 14 Days per Route

Now we score from the latest available date and generate route-level quantile forecasts (`p50`, `q10`, `q90`) for the coming 14 days.


In [ ]:
latest_pos = len(pivot) - 1
latest_date = pivot.index[latest_pos]

forecast_rows: list[dict] = []
for route in pivot.columns:
    context = pivot[route].iloc[latest_pos - cfg.context_len + 1 : latest_pos + 1].to_numpy(np.float32)
    if len(context) != cfg.context_len:
        continue

    start_date = pivot.index[latest_pos - cfg.context_len + 1]
    tfm_pred, tfm_quant = run_timesfm_forecast(context=context, start_date=start_date, horizon=cfg.max_horizon)

    forecast_index = pd.date_range(latest_date + pd.Timedelta(days=1), periods=cfg.max_horizon, freq='D')
    trailing_7_mean = float(context[-7:].mean())

    for i, ts in enumerate(forecast_index):
        p50 = float(tfm_pred[i])
        q10 = float(tfm_quant[i, 1])
        q90 = float(tfm_quant[i, 9])

        forecast_rows.append(
            {
                'date': ts,
                'route': route,
                'timesfm_p50_passengers': p50,
                'timesfm_q10_passengers': q10,
                'timesfm_q90_passengers': q90,
                'uncertainty_band': q90 - q10,
                'demand_index_vs_last7': (p50 - trailing_7_mean) / (trailing_7_mean + 1e-6),
            }
        )

forecast_df = pd.DataFrame(forecast_rows)

print('Latest history date:', latest_date.date())
print('Forecast horizon rows:', len(forecast_df))
print('Routes forecasted:', forecast_df['route'].nunique())
forecast_df.head()

## 7) Convert Forecasts into Airline Operations Plans

This section translates forecasted demand into directly actionable planning tables.

### A) Dynamic Pricing Signals
Simple policy example:

- Demand index >= +10% and uncertainty low -> increase fare by +8%
- Demand index <= -10% -> decrease fare by -6%
- Otherwise -> hold base fare

### B) Fleet Allocation Plan
Estimate aircraft required by route/day from `q90` demand and a seat capacity assumption.

### C) Crew Schedule Plan
Estimate crew pairs from aircraft assignments (simple deterministic staffing ratio).


In [ ]:
# A) Dynamic pricing policy
pricing_df = forecast_df.copy()

conditions = [
    (pricing_df['demand_index_vs_last7'] >= 0.10) & (pricing_df['uncertainty_band'] <= pricing_df['timesfm_p50_passengers'] * 0.35),
    (pricing_df['demand_index_vs_last7'] <= -0.10),
]
choices = [1.08, 0.94]
pricing_df['price_multiplier'] = np.select(conditions, choices, default=1.00)
pricing_df['pricing_action'] = np.select(
    conditions,
    ['raise_fare', 'lower_fare'],
    default='hold_fare',
)

pricing_signals = pricing_df[
    ['date', 'route', 'timesfm_p50_passengers', 'demand_index_vs_last7', 'uncertainty_band', 'price_multiplier', 'pricing_action']
].copy()

pricing_signals.head(12)

In [ ]:
# B) Fleet allocation policy
SEATS_PER_AIRCRAFT = 180
TARGET_LOAD_FACTOR = 0.85

fleet_df = forecast_df.copy()
fleet_df['required_seats_p90'] = np.ceil(fleet_df['timesfm_q90_passengers'] / TARGET_LOAD_FACTOR)
fleet_df['aircraft_needed'] = np.ceil(fleet_df['required_seats_p90'] / SEATS_PER_AIRCRAFT).astype(int)

fleet_allocation = (
    fleet_df.groupby(['date', 'route'], as_index=False)
    .agg(
        expected_passengers=('timesfm_p50_passengers', 'sum'),
        risk_adjusted_passengers_p90=('timesfm_q90_passengers', 'sum'),
        aircraft_needed=('aircraft_needed', 'max'),
    )
    .sort_values(['date', 'aircraft_needed', 'risk_adjusted_passengers_p90'], ascending=[True, False, False])
)

fleet_allocation.head(12)

In [ ]:
# C) Crew scheduling policy
# Assumption: each aircraft assignment needs 2 cockpit + 4 cabin = 6 crew members,
# represented as 2 crew pairs per aircraft-day for planning simplicity.
CREW_PAIRS_PER_AIRCRAFT = 2

crew_schedule = fleet_allocation.copy()
crew_schedule['crew_pairs_needed'] = crew_schedule['aircraft_needed'] * CREW_PAIRS_PER_AIRCRAFT
crew_schedule['crew_members_needed'] = crew_schedule['crew_pairs_needed'] * 2

# Route-level growth signal for network planning.
recent_actual = (
    pivot.tail(7).sum(axis=0).rename('last_7d_actual_passengers').reset_index().rename(columns={'index': 'route'})
)
next_14 = (
    forecast_df.groupby('route', as_index=False)['timesfm_p50_passengers']
    .sum()
    .rename(columns={'timesfm_p50_passengers': 'next_14d_forecast_passengers'})
)
route_priority = recent_actual.merge(next_14, on='route', how='inner')
route_priority['actual_daily_avg'] = route_priority['last_7d_actual_passengers'] / 7.0
route_priority['forecast_daily_avg'] = route_priority['next_14d_forecast_passengers'] / 14.0
route_priority['daily_growth_rate'] = (
    (route_priority['forecast_daily_avg'] - route_priority['actual_daily_avg'])
    / (route_priority['actual_daily_avg'] + 1e-6)
)
route_priority['planning_signal'] = np.select(
    [
        route_priority['daily_growth_rate'] >= 0.12,
        route_priority['daily_growth_rate'] <= -0.12,
    ],
    ['expand_capacity', 'monitor_decline'],
    default='maintain',
)
route_priority = route_priority.sort_values('daily_growth_rate', ascending=False)

crew_schedule.head(12), route_priority.head(12)

## 8) Save Deliverables

Files written:

- `backtest_metrics.csv`
- `backtest_detail.csv`
- `forecast_next_14_days_route.csv`
- `dynamic_pricing_signals.csv`
- `fleet_allocation_plan.csv`
- `crew_schedule_plan.csv`
- `route_priority_for_planning.csv`


In [ ]:
backtest_metrics_path = ART_DIR / 'backtest_metrics.csv'
backtest_detail_path = ART_DIR / 'backtest_detail.csv'
forecast_path = ART_DIR / 'forecast_next_14_days_route.csv'
pricing_path = ART_DIR / 'dynamic_pricing_signals.csv'
fleet_path = ART_DIR / 'fleet_allocation_plan.csv'
crew_path = ART_DIR / 'crew_schedule_plan.csv'
priority_path = ART_DIR / 'route_priority_for_planning.csv'

backtest_metrics.to_csv(backtest_metrics_path, index=False)
backtest_detail.to_csv(backtest_detail_path, index=False)
forecast_df.to_csv(forecast_path, index=False)
pricing_signals.to_csv(pricing_path, index=False)
fleet_allocation.to_csv(fleet_path, index=False)
crew_schedule.to_csv(crew_path, index=False)
route_priority.to_csv(priority_path, index=False)

print('Saved artifacts:')
for p in [
    backtest_metrics_path,
    backtest_detail_path,
    forecast_path,
    pricing_path,
    fleet_path,
    crew_path,
    priority_path,
]:
    print('-', p)

backtest_metrics

## Assumptions, Limitations, and Production Next Steps

### Assumptions in this tutorial

- Passenger demand proxy is built from booking-linked passenger-flight counts.
- Top 30 routes are modeled for runtime practicality in a notebook.
- Pricing/fleet/crew policies are deterministic heuristics layered on forecast outputs.

### Limitations

- Limited historical span in this dataset (~2 months) reduces long-seasonality learning.
- No explicit external regressors (fares, promotions, holidays, weather, events, disruptions).
- Route-specific constraints (slot limits, aircraft turn-around, duty-time legality) are simplified.

### Production upgrades

- Retrain/recalibrate regularly with longer history and richer covariates.
- Add hierarchical reconciliation (airport-region-network).
- Optimize pricing and fleet assignment with constrained optimization (OR-Tools/MILP).
- Wire to monitoring: drift, forecast error by route segment, and decision KPI impact.
